In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
# Define the structure
schema_participantes = StructType([
    StructField("id", IntegerType(), False),      # Cannot be null
    StructField("name", StringType(), True),      # Can be null
    StructField("salary", DoubleType(), True)
])

In [0]:
dicionario_dados_path = '/Volumes/databricks-repo/enem/enem2025/DICIONÁRIO/Dicionário_Microdados_Enem_2025.xlsx'

dicionario_dados = spark.read.excel(dicionario_dados_path)

dicionario_dados = dicionario_dados.withColumn("index", F.monotonically_increasing_id())

window_spec = (
    Window.orderBy("index")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

dicionario_dados = (
    dicionario_dados
    .withColumn('_c0', F.last('_c0', ignorenulls=True).over(window_spec))
    .withColumn('_c1', F.last('_c1', ignorenulls=True).over(window_spec))
    )

display(dicionario_dados)

In [0]:
df = spark.read.table('`databricks-repo`.enem.itens_prova')

In [0]:
df_participantes = spark.read.table('`databricks-repo`.enem.participantes')

df_participantes = (
    df_participantes
        .withColumnRenamed('NU_INSCRICAO', 'id')
        .withColumnRenamed("CO_MUNICIPIO_PROVA",'num_muninicipio')
        .withColumnRenamed("NO_MUNICIPIO_PROVA",'muninicipio')
        .withColumnRenamed("CO_UF_PROVA",'uf')
        .withColumnRenamed("NU_ANO",'ano')
        .withColumnRenamed("TP_SEXO",'sexo')
        .withColumnRenamed("TP_FAIXA_ETARIA",'faixa_etaria')
        .withColumnRenamed("TP_ESTADO_CIVIL",'estado_civil')
        .withColumnRenamed("TP_COR_RACA",'cor')
        .withColumnRenamed("TP_NACIONALIDADE",'nacionalidade')
        .withColumnRenamed("TP_ST_CONCLUSAO",'situacao_conclusao')
        .withColumnRenamed("TP_ANO_CONCLUIU",'ano_conclusao')
        .withColumnRenamed("TP_ENSINO",'ensino')
        .withColumnRenamed("IN_TREINEIRO",'treineiro')
)
df_participantes.show(10)